In [21]:
import warnings
from tqdm import TqdmWarning
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=TqdmWarning)
warnings.filterwarnings("ignore", message="Detected IPython.*")

import pandas as pd
from pathlib import Path
import re
import os
import json
import joblib
import numpy as np
import pyarrow.parquet as pq
import networkx as nx
from sklearn.model_selection import GridSearchCV
import utils as ut
import model_wrappers as mw
from hyperparameter import model_configs
from pathlib import Path


current_path = Path(__file__).resolve().parent if '__file__' in globals() else Path().resolve()



In [22]:

base_path = current_path.parent / "data" / "silver" / "evaluation_benchmark"

# List all subdirectories in base_path
folders = [f for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f))]

# Create a dictionary with folder name as key and full path as value
folder_dict = {f: base_path / f for f in folders}

save_path = current_path.parent / "data" / "gold" / "evaluation_benchmark"
save_path.mkdir(parents=True, exist_ok=True)

# Model training

In [27]:
import time

results_dfs = []

for folder_name, folder_path in folder_dict.items():
    print(f"\nProcessing dataset: {folder_name}")
    symbolic_equations = {"causal": {}, "traditional": {}}
    trained_models = {}
    best_params = {}
    results = []

    # Paths to experimental data
    train_path = folder_path / "train.csv"
    test_path = folder_path / "test.csv"
    intervention_path = folder_path / "interventions.json"
    adj_matrix_path = folder_path / "adj_matrix.csv"
    with open(intervention_path, "r") as f:
        interventions = json.load(f)["environments"]

    # Load train/test data without headers to determine column names
    temp_train_df = pd.read_csv(train_path, header=None)
    n_columns = temp_train_df.shape[1]
    column_names = [f"X{i}" for i in range(n_columns)]
    target_column = 'X' + str(interventions[0]['effect_idxs'][0]-1)

    # Reload with column names
    train_df = pd.read_csv(train_path, header=None, names=column_names)
    test_df = pd.read_csv(test_path, header=None, names=column_names)

    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]
    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]

    # Combine reference + test into one dataset
    combined_data = []
    for env in interventions:
        test_data = pd.DataFrame(env["test_data"], columns=column_names)
        ref_data = pd.DataFrame(env["reference_data"], columns=column_names)
        combined = pd.concat([test_data, ref_data], ignore_index=True)
        combined_data.append(combined)

    inter_df = pd.concat(combined_data)
    X_int = inter_df.drop(columns=[target_column])
    y_int = inter_df[target_column]

    # Load causal graph adjacency matrix
    adj_matrix = pd.read_csv(adj_matrix_path, header=None).values

    G = nx.DiGraph()
    for i, src in enumerate(column_names):
        for j, tgt in enumerate(column_names):
            if adj_matrix[i, j] == 1:
                G.add_edge(src, tgt)

    # Train and evaluate traditional and causal
    for name, cfg in model_configs.items():
        ModelClass = cfg["model_class"]
        params = cfg.get("params", {})
        param_grid = cfg.get("param_grid", None)

        # --- Causal version ---
        t0 = time.perf_counter()
        causal_models = ut.train_causal_models(train_df, G, ModelClass,
                                            model_params=params, param_grid=param_grid)
        causal_train_time = time.perf_counter() - t0

        causal_preds = ut.predict_causal(test_df, G, causal_models, what_if=False)
        rmse_c, wape_c, mae_c = ut.evaluate_metrics(test_df[target_column], causal_preds[target_column], normalize=True)
        results.append(["Causal", name, rmse_c, wape_c, mae_c, "Test", causal_train_time, causal_train_time])

        if name == "SymbolicRegression":
            symbolic_equations["causal"][name] = {}
            for node, model in causal_models.items():
                try:
                    equation = str(model.model.get_best()['equation'])
                    equation = ut.round_numbers_in_string(equation)
                    symbolic_equations["causal"][name][node] = equation
                    print(f"[Causal {name}] Node '{node}': {equation}")
                except:
                    symbolic_equations["causal"][name][node] = "No equation available"

        # --- Traditional version ---
        if param_grid:
            base_model = ModelClass(**params)
            grid_search = GridSearchCV(base_model, param_grid=param_grid,
                                    cv=4, n_jobs=-1, scoring='neg_mean_absolute_error')
            t0 = time.perf_counter()
            grid_search.fit(X_train, y_train)
            trad_total_time = time.perf_counter() - t0
            best_params[name] = grid_search.best_params_
            print(f"[GridSearchCV] Best params for Traditional '{name}': {grid_search.best_params_}")

            # Refit with best params only (single fit)
            t0 = time.perf_counter()
            model = ModelClass(**{**params, **grid_search.best_params_})
            model.fit(X_train, y_train)
            trad_final_time = time.perf_counter() - t0
        else:
            t0 = time.perf_counter()
            model = ModelClass(**params)
            model.fit(X_train, y_train)
            trad_final_time = time.perf_counter() - t0
            trad_total_time = trad_final_time  # No CV, same time
            best_params[name] = params

        y_pred = model.predict(X_test)
        rmse_t, wape_t, mae_t = ut.evaluate_metrics(y_test, y_pred, normalize=True)
        results.append(["Traditional", name, rmse_t, wape_t, mae_t, "Test", trad_total_time, trad_final_time])

        if name == "SymbolicRegression":
            try:
                equation = str(model.model.get_best()['equation'])
                equation = ut.round_numbers_in_string(equation)
                symbolic_equations["traditional"][name] = equation
                print(f"[Traditional {name}] Target equation: {equation}")
            except:
                symbolic_equations["traditional"][name] = "No equation available"

        # Evaluate interventions
        causal_preds_int = ut.predict_causal(inter_df, G, causal_models, what_if=True)
        rmse_c_int, wape_c_int, mae_c_int = ut.evaluate_metrics(y_int, causal_preds_int[target_column], normalize=True)
        results.append(["Causal", name, rmse_c_int, wape_c_int, mae_c_int, "Intervention", causal_train_time, causal_train_time])

        y_pred_int = model.predict(X_int)
        rmse_t_int, wape_t_int, mae_t_int = ut.evaluate_metrics(y_int, y_pred_int, normalize=True)
        results.append(["Traditional", name, rmse_t_int, wape_t_int, mae_t_int, "Intervention", trad_total_time, trad_final_time])

        trained_models[name] = {"causal": causal_models, "traditional": model}

    # Results table
    results_df = pd.DataFrame(results, columns=["Model_Type", "Algorithm", "RMSE", "WAPE", "MAE", "Task", "train_time_total_s", "train_time_final_s"])
    results_df = results_df[["Model_Type", "Algorithm", "MAE", "RMSE", "WAPE", "Task", "train_time_total_s", "train_time_final_s"]]
    results_df['experiment'] = folder_name

    results_dfs.append(results_df)

    # Display symbolic equations
    print("\n" + "="*80)
    print("SYMBOLIC REGRESSION EQUATIONS")
    print("="*80)

    if symbolic_equations["causal"]:
        print("\n--- CAUSAL MODELS ---")
        for model_name, node_equations in symbolic_equations["causal"].items():
            print(f"\n{model_name}:")
            for node, equation in node_equations.items():
                print(f"  {node}: {equation}")

    if symbolic_equations["traditional"]:
        print("\n--- TRADITIONAL MODELS ---")
        for model_name, equation in symbolic_equations["traditional"].items():
            print(f"{model_name}: {equation}")

    print("\n" + "="*80)
    print(results_df)

    # --- Save models and best hyperparameters per dataset ---
    folder_save_path = save_path / folder_name
    models_dir = folder_save_path / "models"
    models_dir.mkdir(parents=True, exist_ok=True)

    # Save raw results as parquet
    results_df.to_parquet(folder_save_path / "results.parquet", index=False, compression="snappy")

    # Save per-dataset filtered tables
    fmt_df = results_df.copy()
    fmt_df['Model_Type'] = fmt_df['Model_Type'].replace({'Causal': 'CML', 'Traditional': 'ML'})
    fmt_df['Algorithm'] = fmt_df['Algorithm'].replace({'RandomForest': 'RForest', 'SymbolicRegression': 'Symbolic',
                                                        'XGBRegression': 'XGB', 'LinearRegression': 'Linreg'})
    ind_intervention = fmt_df[fmt_df["Task"] == "Intervention"].drop(columns=["Task", "experiment"]).sort_values("MAE")
    ind_test = fmt_df[fmt_df["Task"] == "Test"].drop(columns=["Task", "experiment"]).sort_values("MAE")
    ind_intervention.to_parquet(folder_save_path / "intervention_results.parquet", index=False, compression="snappy")
    ind_test.to_parquet(folder_save_path / "test_results.parquet", index=False, compression="snappy")


    # Save best hyperparameters as JSON
    params_path = folder_save_path / "best_hyperparameters.json"
    serialisable_params = {
        model_name: {k: (list(v) if isinstance(v, tuple) else v) for k, v in p.items()}
        for model_name, p in best_params.items()
    }
    with open(params_path, "w") as f:
        json.dump(serialisable_params, f, indent=2)


    # Save trained models
    for model_name, model_dict in trained_models.items():
        joblib.dump(model_dict["traditional"], models_dir / f"{model_name}_traditional.pkl")
        for node, causal_model in model_dict["causal"].items():
            joblib.dump(causal_model, models_dir / f"{model_name}_causal_{node}.pkl")




Processing dataset: csuite_nonlin_simpson
[GridSearchCV] Best params for Traditional 'LinearRegression': {'fit_intercept': True, 'positive': False}

SYMBOLIC REGRESSION EQUATIONS

    Model_Type         Algorithm       MAE      RMSE       WAPE          Task  \
0       Causal  LinearRegression  0.213149  0.277490  26.662086          Test   
1  Traditional  LinearRegression  0.197874  0.251470  24.751423          Test   
2       Causal  LinearRegression  0.927998  1.130853  92.799830  Intervention   
3  Traditional  LinearRegression  0.882500  1.066230  88.250002  Intervention   

   train_time_total_s  train_time_final_s             experiment  
0            0.087650            0.087650  csuite_nonlin_simpson  
1            0.037659            0.001285  csuite_nonlin_simpson  
2            0.087650            0.087650  csuite_nonlin_simpson  
3            0.037659            0.001285  csuite_nonlin_simpson  

Processing dataset: csuite_large_backdoor
[GridSearchCV] Best params for Trad

In [24]:
all_results_df = pd.concat(results_dfs, ignore_index=True)

# Group by Model_Type, Algorithm, and Task, and take the median of the numeric columns
aggregated_results_df = all_results_df.groupby(['Model_Type', 'Algorithm', 'Task']).agg({
    'MAE': 'median',
    'RMSE': 'median',
    'WAPE': 'median',
    'train_time_total_s': 'median',
    'train_time_final_s': 'median',
}).reset_index()


aggregated_results_df


,Model_Type,Algorithm,Task,MAE,RMSE,WAPE,train_time_total_s,train_time_final_s
0,Causal,LinearRegression,Intervention,1.084992,1.318023,108.499153,0.209865,0.209865
1,Causal,LinearRegression,Test,0.284212,0.364265,35.129994,0.209865,0.209865
2,Traditional,LinearRegression,Intervention,1.162874,1.446395,116.287434,0.039477,0.002929
3,Traditional,LinearRegression,Test,0.251024,0.315172,29.792477,0.039477,0.002929


In [25]:
df = aggregated_results_df.copy()

columns_to_display = ['Type', 'Model', 'MAE', 'RMSE']

df['Model_Type'] = df['Model_Type'].replace({'Causal': 'CML', 'Traditional': 'ML'})
df['Algorithm'] = df['Algorithm'].replace({'RandomForest': 'RForest', 'SymbolicRegression': 'Symbolic',
                                           'XGBRegression': 'XGB', 'LinearRegression': 'Linreg',})

df.columns = ["Type", "Model", "Task", "MAE", "RMSE", "WAPE", "train_time_total_s", "train_time_final_s"]

# Filter for Intervention task and drop Task column
intervention_df = df[df["Task"] == "Intervention"].drop(columns=["Task"]).sort_values(by=["MAE"]).copy()

# Filter for Test task and drop Task column
test_df = df[df["Task"] == "Test"].drop(columns=["Task"]).sort_values(by=["MAE"]).copy()

for col in ["MAE", "RMSE", "WAPE", "train_time_total_s", "train_time_final_s"]:
    intervention_df[col] = pd.to_numeric(intervention_df[col], errors="coerce")
    test_df[col] = pd.to_numeric(test_df[col], errors="coerce")

fmt = "{:,.4f}"
fmt_t = "{:,.3f}s"

print("Intervention Results:")

display(intervention_df[columns_to_display].style.format({"MAE": fmt, "RMSE": fmt, "WAPE": fmt,
                                      "train_time_total_s": fmt_t, "train_time_final_s": fmt_t}).hide(axis="index"))

print("\nTest Results:")
display(test_df[columns_to_display].style.format({"MAE": fmt, "RMSE": fmt, "WAPE": fmt,
                               "train_time_total_s": fmt_t, "train_time_final_s": fmt_t}).hide(axis="index"))

display(intervention_df.style.format({"MAE": fmt, "RMSE": fmt, "WAPE": fmt,
                                      "train_time_total_s": fmt_t, "train_time_final_s": fmt_t}).hide(axis="index"))
display(test_df.style.format({"MAE": fmt, "RMSE": fmt, "WAPE": fmt,
                               "train_time_total_s": fmt_t, "train_time_final_s": fmt_t}).hide(axis="index"))

# Save aggregated tables
intervention_df.to_parquet(save_path / "aggregated_intervention_results.parquet", index=False, compression="snappy")
test_df.to_parquet(save_path / "aggregated_test_results.parquet", index=False, compression="snappy")


Intervention Results:


Type,Model,MAE,RMSE
CML,Linreg,1.0850,1.3180
ML,Linreg,1.1629,1.4464



Test Results:


Type,Model,MAE,RMSE
ML,Linreg,0.2510,0.3152
CML,Linreg,0.2842,0.3643


Type,Model,MAE,RMSE,WAPE,train_time_total_s,train_time_final_s
CML,Linreg,1.0850,1.3180,108.4992,0.210s,0.210s
ML,Linreg,1.1629,1.4464,116.2874,0.039s,0.003s


Type,Model,MAE,RMSE,WAPE,train_time_total_s,train_time_final_s
ML,Linreg,0.2510,0.3152,29.7925,0.039s,0.003s
CML,Linreg,0.2842,0.3643,35.1300,0.210s,0.210s


# Individual results for the datasets

In [26]:
all_results_df = pd.concat(results_dfs, ignore_index=True)

for experiment in all_results_df['experiment'].unique():

    print(experiment)
    df = all_results_df[all_results_df['experiment'] == experiment].copy()

    df['Model_Type'] = df['Model_Type'].replace({'Causal': 'CML', 'Traditional': 'ML'})
    df['Algorithm'] = df['Algorithm'].replace({'RandomForest': 'RForest', 'SymbolicRegression': 'Symbolic',
                                            'XGBRegression': 'XGB', 'LinearRegression': 'Linreg',})

    df.columns = ["Type", "Model", "MAE", "RMSE", "WAPE", "Task", "train_time_total_s", "train_time_final_s", "experiment"]

    # Filter for Intervention task and drop Task column
    intervention_df = df[df["Task"] == "Intervention"].drop(columns=["Task", "experiment"]).sort_values(by=["MAE"]).copy()
    test_df = df[df["Task"] == "Test"].drop(columns=["Task", "experiment"]).sort_values(by=["MAE"]).copy()

    for col in ["MAE", "RMSE", "WAPE"]:
        intervention_df[col] = pd.to_numeric(intervention_df[col], errors="coerce")
        test_df[col] = pd.to_numeric(test_df[col], errors="coerce")

    fmt = "{:,.4f}"

    print("Intervention Results:")
    display(intervention_df.style.format({"MAE": fmt, "RMSE": fmt, "WAPE": fmt}).hide(axis="index"))

    print("\nTest Results:")
    display(test_df.style.format({"MAE": fmt, "RMSE": fmt, "WAPE": fmt}).hide(axis="index"))


csuite_nonlin_simpson
Intervention Results:


Type,Model,MAE,RMSE,WAPE,train_time_total_s,train_time_final_s
ML,Linreg,0.8825,1.0662,88.2500,0.040127,0.002710
CML,Linreg,0.9280,1.1309,92.7998,0.122261,0.122261



Test Results:


Type,Model,MAE,RMSE,WAPE,train_time_total_s,train_time_final_s
ML,Linreg,0.1979,0.2515,24.7514,0.040127,0.002710
CML,Linreg,0.2131,0.2775,26.6621,0.122261,0.122261


csuite_large_backdoor
Intervention Results:


...

Type,Model,MAE,RMSE,WAPE,train_time_total_s,train_time_final_s
ML,Linreg,0.2380,0.3002,31.0756,0.038615,0.003149
CML,Linreg,0.2495,0.3174,32.5686,0.297469,0.297469


csuite_symprod_simpson
Intervention Results:


Type,Model,MAE,RMSE,WAPE,train_time_total_s,train_time_final_s
CML,Linreg,1.0993,1.3409,109.9251,0.084025,0.084025
ML,Linreg,1.1129,1.4039,111.2914,0.038827,0.002686



Test Results:


Type,Model,MAE,RMSE,WAPE,train_time_total_s,train_time_final_s
ML,Linreg,0.2640,0.3302,28.5094,0.038827,0.002686
CML,Linreg,0.3490,0.4118,37.6914,0.084025,0.084025
